<a href="https://colab.research.google.com/github/Ruturaj2472/RAG-HybridSearch_Rerank_EndToEnd/blob/main/RAG_Hybrid_Search_Rerank_EndToEnd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End RAG Pipeline: Hybrid Search + Reranking + Generation
### LangChain · Cohere · ChromaDB · BM25

---


```
Indexing:   PDFs -> Text Extraction -> Chunking -> Cohere Embeddings -> Chroma (dense) + BM25 (sparse)
                                                                              |
Querying:   Question -> Hybrid Retriever (BM25 + Chroma) -> Cohere Reranker -> Top-K Context -> LLM -> Answer
```

## Demo Documents

| Paper | Topic |
|-------|-------|
| [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762) | Transformer architecture |
| [You Only Look Once — YOLO (Redmon et al., 2015)](https://arxiv.org/abs/1506.02640) | Real-time object detection |

> Upload `1706.03762v7.pdf` and `1506.02640v5.pdf` to `/content/` in Colab before running (or swap in your own PDFs).


## 1. Setup & Installation


In [1]:
# Install all required packages:
# - langchain-cohere      : Cohere LLM, Embeddings, and Reranker integration for LangChain
# - langchain             : Core LangChain framework (chains, prompts, runnables)
# - langchain-classic     : Houses EnsembleRetriever / ContextualCompressionRetriever in newer LangChain versions
# - pdfminer.six          : Extracts raw text from PDF files
# - chromadb              : Vector database used for dense (semantic) retrieval
# - rank_bm25             : Backend used by BM25Retriever for sparse (keyword) retrieval
# - langchain-community   : Community-maintained retrievers/vectorstores (BM25Retriever, Chroma wrapper)
# - langchain-text-splitters : RecursiveCharacterTextSplitter for chunking documents
!pip install langchain-cohere langchain langchain-classic pdfminer.six chromadb rank_bm25 langchain-community langchain-text-splitters -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 

**Package roles at a glance**

- `langchain-cohere` -> `ChatCohere` (generation), `CohereEmbeddings` (dense vectors), `CohereRerank` (reranker)
- `chromadb` + `langchain-community` -> dense vector store (`Chroma`)
- `rank_bm25` + `langchain-community` -> sparse keyword retriever (`BM25Retriever`)
- `langchain-classic` -> `EnsembleRetriever` (hybrid fusion) and `ContextualCompressionRetriever` (reranking wrapper)


## 2. Imports & API Key Configuration


In [2]:
import os
import pandas as pd
from google.colab import userdata

# Set the Cohere API key as an environment variable.
# ChatCohere / CohereEmbeddings / CohereRerank all read the key from this env var automatically.
os.environ["COHERE_API_KEY"] = userdata.get('COHERE_KEY')


In [3]:
# --- Core LangChain building blocks ---
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# --- Document loading / chunking ---
from pdfminer.high_level import extract_text as extract_text_pdf_miner
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Vector store + retrievers (Hybrid Search) ---
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# --- Reranking ---
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

# --- Cohere embeddings + generation model ---
from langchain_cohere import CohereEmbeddings, ChatCohere


/tmp/ipykernel_899/486851553.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


## 3. Embeddings & Document Ingestion


- `embed-english-v3.0` Cohere embedding model for the dense (Chroma) side
- `RecursiveCharacterTextSplitter` with `chunk_size=2048`, `chunk_overlap=512`
- A single loader function that routes documents to **either** the BM25 retriever (`source=1`) **or** the Chroma vector store (`source=2`)

In [4]:
# Directory where the Chroma vector database will persist its data
persist_directory = "/content/hybrid-search"

# Initialize Cohere embeddings with the specified model
# "embed-english-v3.0" is a pre-trained English language embedding model by Cohere
# The user_agent parameter specifies the tool/library using the Cohere API (LangChain here)
embedding = CohereEmbeddings(
    model="embed-english-v3.0",
    user_agent="langchain"
)

print("Cohere embedding model initialized:", embedding.model)


Cohere embedding model initialized: embed-english-v3.0


In [5]:
def load_data_to_vectordb(file_path, source):
    """
    Extract text from a PDF, chunk it, and route the chunks into the
    appropriate retrieval backend.

    """
    global bm25_retriever
    pages = []

    for pdf_name in [file_path]:
        # Open the PDF in binary mode and extract raw text
        with open(pdf_name, 'rb') as f:
            text = extract_text_pdf_miner(f)

            # Clean the extracted text by collapsing newlines into a single string
            cleaned_text = " ".join(text.split("\n"))

            docs = []

            # Split into overlapping chunks so context is not lost at chunk boundaries
            splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

            for chunk in splitter.split_text(cleaned_text):
                docs.append(Document(page_content=chunk, metadata={"retrived_from": source, "source": pdf_name}))
                pages.append(Document(page_content=chunk, metadata={"retrived_from": source, "source": pdf_name}))

        # source == 1 -> build the sparse BM25 retriever from these chunks
        # otherwise    -> push chunks into the persistent Chroma vector store
        if source == 1:
            bm25_retriever = BM25Retriever.from_documents(pages)
        else:
            db = Chroma.from_documents(
                documents=docs,
                persist_directory=persist_directory,
                embedding=embedding
            )

# Ingest both demo PDFs: one feeds BM25, the other feeds Chroma
load_data_to_vectordb(file_path="/content/1706.03762v7.pdf", source=1)
load_data_to_vectordb(file_path="/content/1506.02640v5.pdf", source=2)

print("Ingestion complete: BM25 retriever built and Chroma vector store populated.")


Ingestion complete: BM25 retriever built and Chroma vector store populated.


## 4. Hybrid Search (BM25 + Dense Retrieval)

We combine a **sparse keyword retriever (BM25)** with a **dense semantic retriever (Chroma)** using LangChain's `EnsembleRetriever`, which fuses both rankings using weighted Reciprocal Rank Fusion.


In [6]:
# Configure the BM25 retriever to return the top 2 keyword matches
bm25_retriever.k = 2
print("BM25 retriever type:", type(bm25_retriever))


BM25 retriever type: <class 'langchain_community.retrievers.bm25.BM25Retriever'>


In [7]:
# Reconnect to the persisted Chroma vector store and wrap it as a retriever
docsearch = Chroma(persist_directory=persist_directory, embedding_function=embedding)
retriever_chromadb = docsearch.as_retriever(search_kwargs={"k": 5})

# Fuse BM25 (sparse) + Chroma (dense) into a single hybrid retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever_chromadb], weights=[0.3, 0.7]
)

print("Hybrid (ensemble) retriever ready.")


Hybrid (ensemble) retriever ready.


/tmp/ipykernel_899/3422843966.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  docsearch = Chroma(persist_directory=persist_directory, embedding_function=embedding)


In [8]:
# --- Validation: run a sample query through the hybrid retriever ---
query = "What is Self-Attention in Transformers?"

hybrid_docs = ensemble_retriever.invoke(query)

print(f"Retrieved {len(hybrid_docs)} documents for query: '{query}'\n")
hybrid_docs


Retrieved 7 documents for query: 'What is Self-Attention in Transformers?'



[Document(metadata={'retrived_from': 2, 'source': '/content/1506.02640v5.pdf'}, page_content='they are, and how they inter- act. The human visual system is fast and accurate, allow- ing us to perform complex tasks like driving with little con- scious thought. Fast, accurate algorithms for object detec- tion would allow computers to drive cars without special- ized sensors, enable assistive devices to convey real-time scene information to human users, and unlock the potential for general purpose, responsive robotic systems.  Current detection systems repurpose classiﬁers to per- form detection. To detect an object, these systems take a classiﬁer for that object and evaluate it at various locations and scales in a test image. Systems like deformable parts models (DPM) use a sliding window approach where the classiﬁer is run at evenly spaced locations over the entire image [10].  Figure 1: The YOLO Detection System. Processing images with YOLO is simple and straightforward. Our system (1)

In [9]:
# Display retrieved chunks in a tidy DataFrame for easier inspection
retrieval_df = pd.DataFrame()

page_content, retrieval_source, pdf_source = [], [], []
for doc in hybrid_docs:
    page_content.append(doc.page_content)
    retrieval_source.append(doc.metadata['retrived_from'])
    pdf_source.append(doc.metadata['source'])

retrieval_df['page_content'] = page_content
retrieval_df['retrieval_source'] = retrieval_source   # 1 = BM25 origin, 2 = Chroma origin
retrieval_df['pdf_source'] = pdf_source

retrieval_df.head(10)


,page_content,retrieval_source,pdf_source
0,"they are, and how they inter- act. The human v...",2,/content/1506.02640v5.pdf
1,Search [35] generates potential bounding boxes...,2,/content/1506.02640v5.pdf
2,want one bounding box predictor to be responsi...,2,/content/1506.02640v5.pdf
3,"You Only Look Once: Uniﬁed, Real-Time Object D...",2,/content/1506.02640v5.pdf
4,to examine the accuracy-performance tradeoffs ...,2,/content/1506.02640v5.pdf
5,identically. This consists of two linear trans...,1,/content/1706.03762v7.pdf
6,our model contains no recurrence and no convol...,1,/content/1706.03762v7.pdf


## 5. Reranking the Hybrid Results

Hybrid search gives us a good *candidate* set, but BM25 and dense-similarity scores aren't directly comparable, so the fused order is only approximate.

We attach Cohere's **`CohereRerank`** model on top of the hybrid retriever via `ContextualCompressionRetriever`. This re-scores every candidate the hybrid retriever returns using a cross-encoder-style relevance model, giving a much more reliable final ranking before we hand context to the LLM.


In [10]:
# Initialize the Cohere reranker
compressor = CohereRerank(model='rerank-v4.0-pro')

# Wrap the HYBRID retriever with reranking:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever
)

print("Reranking-enabled hybrid retriever (compression_retriever) ready.")


Reranking-enabled hybrid retriever (compression_retriever) ready.


In [11]:
# --- Validation: compare hybrid-only ordering vs. hybrid + reranked ordering ---
validation_query = "What is the architecture of transformers?"

# 1) Hybrid search alone (no reranking)
non_rerank_df = pd.DataFrame(columns=['Text', 'source', 'retrieval_source'])
pre_rerank_docs = ensemble_retriever.invoke(validation_query)

for doc in pre_rerank_docs[:3]:
    non_rerank_df = non_rerank_df._append(
        {
            'Text': doc.page_content,
            'source': doc.metadata['source'],
            'retrieval_source': doc.metadata['retrived_from']
        },
        ignore_index=True
    )

print("Top 3 results — Hybrid search WITHOUT reranking:")
non_rerank_df.head(3)


Top 3 results — Hybrid search WITHOUT reranking:


,Text,source,retrieval_source
0,predicted box ﬁts the object. Figure 2: The M...,/content/1506.02640v5.pdf,2
1,3: The Architecture. Our detection network has...,/content/1506.02640v5.pdf,2
2,small error in a small box has a much greater ...,/content/1506.02640v5.pdf,2


In [12]:
# 2) Hybrid search + reranking
source_df = pd.DataFrame(columns=['Text', 'source', 'relevance_score'])
reranked_docs = compression_retriever.invoke(validation_query)

for doc in reranked_docs[:3]:
    source_df = source_df._append(
        {
            'Text': doc.page_content,
            'source': doc.metadata['source'],
            'relevance_score': doc.metadata['relevance_score']
        },
        ignore_index=True
    )

print("Top 3 results — Hybrid search WITH reranking:")
source_df.head(3)


Top 3 results — Hybrid search WITH reranking:


/tmp/ipykernel_899/3715644444.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  source_df = source_df._append(


,Text,source,relevance_score
0,mechanism instead of sequence- aligned recurre...,/content/1706.03762v7.pdf,0.961040
1,as state of the art approaches in sequence mod...,/content/1706.03762v7.pdf,0.866856
2,3: The Architecture. Our detection network has...,/content/1506.02640v5.pdf,0.443215


Notice that the **relevance_score** column only appears after reranking, and the ordering/content of the top results can shift compared to raw hybrid search — reranking is directly optimizing for query relevance rather than approximate keyword/embedding similarity.


## 6. Generation Model



In [13]:
# Initialize the LLM using Cohere's chat model
llm = ChatCohere(model="command-a-plus-05-2026", temperature=0)

prompt_str = """Answer the question below using the context:

Context: {context}

Question: {question}

Answer: """

prompt = ChatPromptTemplate.from_template(prompt_str)

# Output parser converts the LLM's ChatMessage response into a plain string
output_parser = StrOutputParser()

print("Generation model, prompt template, and output parser ready.")


Generation model, prompt template, and output parser ready.


## 7. End-to-End RAG Chain (Hybrid Search -> Rerank -> Generate)

We now wire everything together into a single LangChain **Runnable** pipeline:

```
question --> compression_retriever (hybrid + rerank) --> context
question -----------------------------------------------> question
{context, question} --> prompt --> llm --> output_parser --> final answer
```


In [14]:
def format_docs(docs):
    """Flatten a list of retrieved Documents into a single context string for the prompt."""
    return "\n\n".join(doc.page_content for doc in docs)

# Build the retrieval stage:
# - "context"  -> runs the query through the hybrid + reranking retriever, then formats it to text
# - "question" -> passes the original question through unchanged
retrieval = RunnableParallel(
    {
        "context": compression_retriever | format_docs,
        "question": RunnablePassthrough()
    }
)

# Full end-to-end chain: Hybrid Search -> Rerank -> Prompt -> LLM -> Parse output
chain = retrieval | prompt | llm | output_parser

print("End-to-end RAG chain assembled: retrieval | prompt | llm | output_parser")


End-to-end RAG chain assembled: retrieval | prompt | llm | output_parser


## 8. Demo & Validation


### 8.1 `.invoke()` — single question, single answer


In [15]:
response = chain.invoke("What is Self-Attention in Transformers?")

print("Question: What is Self-Attention in Transformers?\n")
print("Answer:\n", response)


Question: What is Self-Attention in Transformers?

Answer:
 Self‑Attention is a layer that lets every position in a sequence attend to every other position. For each token it computes a weighted sum of all token representations (including itself) based on learned attention weights, producing a context vector that captures information from the whole sequence. This enables the model to capture long‑range dependencies and relative positional relationships without recurrence or convolution. The operation has a total computational complexity of O(n²·d) and can be fully parallelized (no sequential steps). In the Transformer architecture, self‑attention layers are placed at the bottom of both the encoder and decoder stacks, and they are combined with positional encodings (sine/cosine functions) to inject order information into the embeddings.


### 8.2 `.batch()` — multiple questions in parallel


In [16]:
questions = [
    "What is YOLO?",
    "How is Transformer different from YOLO?"
]

batch_responses = chain.batch(questions)

for q, r in zip(questions, batch_responses):
    print("Question:", q)
    print("Answer:", r)
    print("\n" + "-" * 80 + "\n")


Question: What is YOLO?
Answer: YOLO (You Only Look Once) is a unified, real‑time object detection system. Instead of using a separate classifier and region‑proposal pipeline, it frames detection as a single regression problem: a single convolutional neural network takes a full image as input and directly outputs spatially separated bounding boxes together with the probabilities that each box contains objects of various classes. Because the entire detection pipeline is a single network, it can be trained end‑to‑end to optimize detection performance. This architecture makes YOLO extremely fast (e.g., 45 fps for the base model and >150 fps for a smaller Fast YOLO version) while still achieving high accuracy and generalizing well to new domains.

--------------------------------------------------------------------------------

Question: How is Transformer different from YOLO?
Answer: The provided context does not contain any information about Transformer, so we cannot answer the question 

### 8.3 `.stream()` — token-by-token streaming output


In [17]:
print("Question: What are the 3 vectors in the Transformer architecture?\n")
print("Streaming answer:\n")
for chunk in chain.stream("What are the 3 vectors in the Transformer architecture?"):
    print(chunk, flush=True, end="")


Question: What are the 3 vectors in the Transformer architecture?

Streaming answer:

In the Transformer, each position’s representation is projected into three distinct vectors that are used in the attention operation:

- **Query (Q)** – represents the position that is querying information from others.  
- **Key (K)** – represents the position that can be queried; compatibility is measured by the dot product between Q and K.  
- **Value (V)** – holds the information that is retrieved when a query matches a key.

These three vectors (Q, K, V) are computed via linear transformations of the input embeddings (combined with positional encodings) and are the core components of the scaled dot‑product attention mechanism.

---
## Summary

| Stage | Component | Source Notebook |
|---|---|---|
| Ingestion | `extract_text_pdf_miner`, `RecursiveCharacterTextSplitter` | Retrieval Optimization |
| Sparse retrieval | `BM25Retriever` | Retrieval Optimization |
| Dense retrieval | `Chroma` + `CohereEmbeddings` | Retrieval Optimization |
| Hybrid fusion | `EnsembleRetriever` (weights `[0.3, 0.7]`) | Retrieval Optimization |
| Reranking | `CohereRerank` (`rerank-v4.0-pro`) + `ContextualCompressionRetriever` | Retrieval Optimization |
| Generation | `ChatCohere` (`command-a-plus-05-2026`) | Personal Resource Assistant |
| Orchestration | `RunnableParallel` \| `ChatPromptTemplate` \| LLM \| `StrOutputParser` | Personal Resource Assistant |

The key change from the two source notebooks: the **reranker's `base_retriever` is the hybrid `ensemble_retriever`** (not a plain Chroma retriever), so every answer the LLM sees is grounded in reranked, hybrid-searched context — a full retrieval -> rerank -> generate pipeline in one flow.
